In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import diopy
import bbknn
import scanorama
import scvi
import scrublet as scr
import scanpy.external as sce
import gc

sc.logging.print_header()


In [ ]:
# data dir
input_dir = " "
output_dir = " "
file_list = os.listdir(input_dir)
# file_list = ["BRCA"]

In [ ]:
file_list

In [ ]:
## pre work
cluster_method = "leiden"
batch_method = "bbknn"
regress = True
resolution = 3
random_state = 123


In [ ]:
# data run
for file in file_list:

    if file != "scrublet_stat.csv":
        print(f"################################## {file} - {batch_method} process ##################################")

        ## data read
        # scRNA_current = diopy.input.read_h5(file = f"{input_dir}/{file}/scRNA_QC.h5")
        scRNA_current = sc.read_h5ad(f"{input_dir}/{file}/scRNA_QC.h5ad")

        ## dir create
        output_file = f"{output_dir}/{file}"
        os.makedirs(output_file, exist_ok=True)

        ## combat
        if batch_method == "combat":
            sc.pp.combat(scRNA_current, key='sample_ID')

        ## data nor
        # scRNA_current.raw = scRNA_current
        scRNA_current.layers["counts"] = scRNA_current.X.copy()
        sc.pp.normalize_total(scRNA_current, target_sum=1e4)
        sc.pp.log1p(scRNA_current)
        scRNA_current.layers["log1p"] = scRNA_current.X.copy()
        scRNA_current.raw = scRNA_current

        ## HVG
        sc.pp.highly_variable_genes(scRNA_current, flavor='seurat',batch_key= "sample_ID", n_top_genes=3000)
        scRNA_current = scRNA_current[:, scRNA_current.var.highly_variable]

        ## regress out effects of total counts per cell and the percentage of mitochondrial genes expressed
        if regress == True:
            sc.pp.regress_out(scRNA_current, ['nCount_RNA','percent.mt'])

        ## data scale
        sc.pp.scale(scRNA_current, max_value=10)

        ## run PCA
        sc.tl.pca(scRNA_current, svd_solver='arpack', use_highly_variable=True, n_comps=50, random_state=random_state)
        # sc.pl.pca_variance_ratio(scRNA_current, n_pcs=50, log=True)

        ## origin copy
        scRNA_origin = scRNA_current.copy()
        sc.pp.neighbors(scRNA_origin)
        sc.tl.leiden(scRNA_origin, resolution=resolution, n_iterations=-1, random_state=random_state)
        scRNA_origin.write_h5ad(f"{output_file}/scRNA_in_batch.h5ad", compression="gzip")

        ## remove batch
        if batch_method == "bbknn":
            sc.external.pp.bbknn(scRNA_current, batch_key = "sample_ID", n_pcs = 50)
        elif batch_method == "scvi":
            scvi.model.SCVI.setup_anndata(scRNA_current, layer="counts", batch_key="sample_ID")
            vae = scvi.model.SCVI(scRNA_current, n_layers=2, n_latent=30, gene_likelihood="nb")
            vae.train()
            scRNA_current.obsm["X_scvi"] = vae.get_latent_representation()
        elif batch_method == "scanorama":
            sc.external.pp.scanorama_integrate(scRNA_current, key="sample_ID", verbose=1)
        elif batch_method == "harmony":
            sc.external.pp.harmony_integrate(scRNA_current, key="sample_ID")
        elif batch_method == "mnn":
            sc.external.pp.mnn_correct(scRNA_current, batch_key = "sample_ID")
        elif batch_method == "none":
            pass


        # data save
        scRNA_current.write_h5ad(f"{output_file}/scRNA_remove_batch.h5ad", compression="gzip")
        diopy.output.write_h5(scRNA_current, file = f"{output_file}/scRNA_remove_batch.h5",save_X=False)

        ## space free
        del scRNA_current
        gc.collect()

    
    